# 🧪 PhARMA RAG — Learning Edition
### Build a Retrieval-Augmented Generation system from scratch, one concept at a time

This is the **study companion** to `PhARMA_RAG_Pipeline.ipynb`. Every section follows the same rhythm:

> **🧠 Concept** — the idea in plain words &nbsp;→&nbsp; **💻 Code** — the implementation &nbsp;→&nbsp; **🔍 What just happened** — reading the output

**The one big idea:** an LLM alone hallucinates and can't cite sources — dangerous for medicine.
RAG splits the job in two so the model may **only speak from trusted, retrieved, cited text**:

```
structured JSON → chunk → preprocess → retrieve (sparse / dense / hybrid) → pack context → prompt → grounded answer → evaluate
```

| Step | Lab | What you'll learn |
|:--|:--|:--|
| 0 | — | Setup (robust, works offline) |
| 1 | 5 | Chunking + **medically-safe** preprocessing |
| 2 | 6 | Evaluation harness + sparse retrieval (BoW, TF-IDF, BM25) |
| 3 | 7 | Dense (semantic) + hybrid retrieval + FAISS |
| 4 | 8 | RAG assembly — context packing + prompt engineering |
| 5 | 9 | Grounded generation with a local LLM (Ollama) |
| 6 | — | Evaluation: groundedness + the refusal test |

**Design rules (why this project is careful):**
- No LangChain / LlamaIndex — everything is raw so you see the mechanics.
- **Preserve negation words and numbers** — `no`, `not`, `500 mg` change clinical meaning.
- Every generated answer must include *"This is not medical advice."*
- `documents → fit_transform()`, `query → transform()` — **never** fit on the query.

> ▶️ **How to run:** `Cell → Run All`. Cells degrade gracefully — no internet, no `rank_bm25`, no FAISS, or no Ollama? The notebook still runs and tells you what was skipped.

---
# Part 0 — Setup & Configuration

We keep every tunable knob in one place so experiments are reproducible. `RANDOM_SEED` fixes randomness; `K` is how many chunks we retrieve per query.

In [ ]:
# If a package is missing, uncomment and run this once (needs internet):
# !pip install -q pandas numpy scikit-learn rank-bm25 sentence-transformers faiss-cpu nltk requests

In [ ]:
import json, os, re, math, warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)

# ── Configuration (one place for every knob) ─────────────────────────
RANDOM_SEED      = 42
K                = 5                      # retrieve top-5 chunks per query
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"     # small, fast sentence-embedding model
OLLAMA_HOST      = "http://localhost:11434"
OLLAMA_MODEL     = "deepseek-r1:1.5b"     # local LLM for generation

np.random.seed(RANDOM_SEED)

# ── Robustly locate data/drugs.json no matter where the notebook runs ─
def find_drugs_json():
    for base in [Path.cwd(), Path.cwd() / "pharma-rag", *Path.cwd().parents]:
        p = base / "data" / "drugs.json"
        if p.exists():
            return p
    raise FileNotFoundError("Could not find data/drugs.json — check your working directory.")

DRUGS_JSON = find_drugs_json()
print("Configuration loaded.")
print(f"  drugs.json  : {DRUGS_JSON}")
print(f"  Embed model : {EMBED_MODEL_NAME}")
print(f"  Ollama model: {OLLAMA_MODEL}")

In [ ]:
# ── NLTK resources (tokenizer, stopwords, lemmatizer). Safe if offline. ──
import nltk
NLTK_OK = True
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        NLTK_OK = False
print(f"NLTK downloads attempted (ok={NLTK_OK}). Fallbacks exist if any failed.")

---
# Part 1 — Lab 5: Chunking & Medically-Safe Preprocessing

## 1.1 Chunking — turn documents into searchable units

### 🧠 Concept
You cannot search a whole drug entry as one blob: a question about *side effects* would drag in
mechanism, dosage, everything. So we **split each document into small, self-contained pieces
("chunks")**. Here the split is natural — each drug has 7 text fields → **12 drugs × 7 fields = 84 chunks**.

Two tricks make chunks powerful:
1. **Self-describing text** — we prefix every chunk with `DrugName — field: ...` so it still makes
   sense when pulled out alone.
2. **Metadata** — we keep `drug`, `category`, `field` columns so we can later *cite* exactly where
   an answer came from.

In [ ]:
# Load the raw structured data
with open(DRUGS_JSON, "r", encoding="utf-8") as f:
    drugs_data = json.load(f)

print(f"Loaded {len(drugs_data)} drugs. Fields per drug: {list(drugs_data[0].keys())}")
print("Drugs:", ", ".join(d["name"] for d in drugs_data))

In [ ]:
# ── Expand each drug into one chunk per text field ──────────────────
FIELD_LABELS = ["description", "mechanism", "indications", "dosage",
                "side_effects", "contraindications", "interactions"]

records = []
for drug in drugs_data:
    for field in FIELD_LABELS:
        text = drug.get(field, "")
        if text:
            records.append({
                "drug":     drug["name"].lower(),
                "category": drug["category"].lower(),
                "field":    field,
                # self-describing chunk: carries its own context
                "text":     f"{drug['name']} — {field}: {text}",
            })

df_chunks = pd.DataFrame(records)
print(f"Expanded {len(df_chunks)} chunks from {len(drugs_data)} drugs "
      f"(expected {len(drugs_data)}×{len(FIELD_LABELS)}={len(drugs_data)*len(FIELD_LABELS)})")
df_chunks.head()

In [ ]:
# ── A little EDA: how long is each chunk? ──
df_chunks["word_count"] = df_chunks["text"].str.split().str.len()
print("Words per chunk:\n", df_chunks["word_count"].describe().round(1).to_string())
print("\nMean words by field:")
print(df_chunks.groupby("field")["word_count"].mean().round(1).sort_values(ascending=False).to_string())

### 🔍 What just happened
`df_chunks` is now our **corpus**: 84 rows, each a searchable unit with its own metadata. Dosage
chunks are the longest (lots of numbers), contraindications the shortest. Every retriever below
searches *these 84 rows* — chunking quality caps everything downstream.

## 1.2 Preprocessing — normalize text, but *safely*

### 🧠 Concept
Before matching words we clean them: lowercase, strip URLs, collapse spaces, optionally drop
common **stopwords** (`the`, `is`, `of`), and reduce words to a root:
- **Stemming** = crude chop (`dosing → dos`) — fast, ugly, can mangle meaning.
- **Lemmatization** = dictionary-aware (`doses → dose`) — slower, keeps real words.

**⚠️ The medical twist — cleaning can be dangerous:**
- Removing numbers destroys `500 mg`.
- Removing stopwords can delete **"no" / "not"** → *"no known interactions"* becomes
  *"known interactions"* — the **opposite** meaning.

So our preprocessor keeps a **protected-negation list** and preserves numbers by default. Every
step is a toggle so we can build different *profiles* and compare them.

In [ ]:
import string

# Tokenizer with fallback (NLTK → plain split)
def safe_word_tokenize(text):
    try:
        return nltk.word_tokenize(text)
    except Exception:
        return text.split()

# Stopwords with fallback (NLTK → sklearn)
try:
    _STOP = set(nltk.corpus.stopwords.words("english"))
except Exception:
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
    _STOP = set(ENGLISH_STOP_WORDS)

from nltk.stem import PorterStemmer, WordNetLemmatizer
_STEMMER, _LEMMATIZER = PorterStemmer(), WordNetLemmatizer()

def safe_lemmatize(tok):
    try:
        return _LEMMATIZER.lemmatize(tok)
    except Exception:
        return tok

# 🔒 Words we must NEVER remove or stem — they flip clinical meaning
PROTECTED_NEGATION = {"no", "not", "nor", "never", "none",
                      "neither", "nobody", "nothing", "nowhere"}

print(f"Stopwords loaded: {len(_STOP)}")
print(f"Protected negations: {PROTECTED_NEGATION}")

In [ ]:
def preprocess_text(text, lowercase=True, remove_url=True, remove_punct=False,
                    remove_num=False, normalize_space=True, remove_stop_words=False,
                    preserve_negation=True, use_stemming=False, use_lemmatization=False):
    """Modular text cleaner. Every step is a switch → lets us build 'profiles'."""
    if not isinstance(text, str) or not text.strip():
        return ""

    if remove_url:   text = re.sub(r"https?://\S+", "", text)
    if lowercase:    text = text.lower()
    if remove_punct: text = text.translate(str.maketrans("", "", string.punctuation))
    if remove_num:   text = re.sub(r"\b\d+\b", "", text)          # ⚠️ destroys "500 mg"
    if normalize_space: text = re.sub(r"\s+", " ", text).strip()

    if remove_stop_words or use_stemming or use_lemmatization:
        tokens = safe_word_tokenize(text)
        if remove_stop_words:
            tokens = [t for t in tokens
                      if t not in _STOP or (preserve_negation and t in PROTECTED_NEGATION)]
        if use_stemming:
            tokens = [_STEMMER.stem(t) if t not in PROTECTED_NEGATION else t for t in tokens]
        if use_lemmatization:
            tokens = [safe_lemmatize(t) if t not in PROTECTED_NEGATION else t for t in tokens]
        text = " ".join(tokens)

    if normalize_space: text = re.sub(r"\s+", " ", text).strip()
    return text

print("preprocess_text() ready.")

### 🧠 Concept — four profiles, one dangerous lesson
We bundle the switches into 4 named **profiles**, from gentle to aggressive. Then we run the same
sentence through all four and *watch the meaning break*.

In [ ]:
PROFILES = {
    "minimal_clean":       dict(lowercase=True, remove_url=True, normalize_space=True),
    "stopword_reduced":    dict(lowercase=True, remove_url=True, normalize_space=True,
                                remove_stop_words=True, preserve_negation=True),
    "aggressive_stemmed":  dict(lowercase=True, remove_url=True, remove_punct=True,
                                remove_num=True, normalize_space=True, remove_stop_words=True,
                                preserve_negation=False, use_stemming=True),
    "readable_lemmatized": dict(lowercase=True, remove_url=True, normalize_space=True,
                                remove_stop_words=True, preserve_negation=True,
                                use_lemmatization=True),
}

demo = ("Paracetamol — side_effects: No known hepatotoxicity at therapeutic doses "
        "(500 mg). Not recommended with warfarin.")
print("Original:\n ", demo, "\n")
for name, kw in PROFILES.items():
    print(f"{name:22s} → {preprocess_text(demo, **kw)}")

### 🔍 What just happened — the safety verdict
Look at `aggressive_stemmed`: *"No known hepatotoxicity ... (500 mg). Not recommended"* becomes
*"known hepatotox ... mg recommend"* — it **deleted the "No", the "Not", and the "500"**. For a
medical system that's catastrophic. That's why our pipeline's default is **`readable_lemmatized`**
(gentle: keeps negation and numbers). *Preprocessing here is a safety decision, not just a tuning
knob.* We now attach the cleaned text as a `clean` column for the lexical retrievers.

In [ ]:
# Default cleaning for the whole corpus (used by TF-IDF / BM25)
df_chunks["clean"] = df_chunks["text"].apply(lambda t: preprocess_text(t, **PROFILES["readable_lemmatized"]))
df_chunks[["drug", "field", "clean"]].head(3)

---
# Part 2 — Lab 6: Evaluation Harness + Sparse Retrieval

## 2.1 Ground truth — you can't improve what you can't measure

### 🧠 Concept
Before building *any* retriever, we define **10 queries whose correct answers we already know**
("ground truth"). Each ground-truth rule is a tiny function that marks which chunks *should* be
returned. This lets us score every retriever objectively and fairly.

In [1]:
QUERY_SPECS = [
    {"query": "What are the side effects of paracetamol?",
     "ground_truth": lambda r: r["drug"] == "paracetamol" and r["field"] == "side_effects"},
    {"query": "What is the mechanism of action of ibuprofen?",
     "ground_truth": lambda r: r["drug"] == "ibuprofen" and r["field"] == "mechanism"},
    {"query": "What are the indications for amoxicillin?",
     "ground_truth": lambda r: r["drug"] == "amoxicillin" and r["field"] == "indications"},
    {"query": "What is the dosage of metformin?",
     "ground_truth": lambda r: r["drug"] == "metformin" and r["field"] == "dosage"},
    {"query": "What drugs interact with warfarin?",
     "ground_truth": lambda r: "warfarin" in r["text"].lower()},
    {"query": "Which drugs can cause hepatotoxicity?",
     "ground_truth": lambda r: "hepatotoxicity" in r["text"].lower() or "liver" in r["text"].lower()},
    {"query": "What are the contraindications for NSAIDs?",
     "ground_truth": lambda r: "nsaid" in r["text"].lower() and r["field"] == "contraindications"},
    {"query": "Which antibiotics are in the dataset?",
     "ground_truth": lambda r: "antibiotic" in r["category"].lower()},
    {"query": "Drugs used for diabetes treatment",
     "ground_truth": lambda r: "diabetes" in r["text"].lower() or "antidiabetic" in r["category"].lower()},
    {"query": "Drugs that affect blood pressure",
     "ground_truth": lambda r: ("hypertension" in r["text"].lower()
                                or "blood pressure" in r["text"].lower()
                                or "vasodilation" in r["text"].lower())},
]

# Turn each rule into the SET of correct chunk indices
for i, spec in enumerate(QUERY_SPECS, 1):
    mask = df_chunks.apply(spec["ground_truth"], axis=1)
    spec["gt_indices"] = set(df_chunks[mask].index.tolist())
    print(f'Q{i:>2}: {spec["query"]:<48s} → {len(spec["gt_indices"])} correct chunk(s)')

NameError: name 'df_chunks' is not defined

## 2.2 Retrieval metrics — four questions about a ranked list

### 🧠 Concept
Given the top-`k` results, we ask four things (`@k` = "looking at the first k"):

| Metric | Question | Cares about |
|:--|:--|:--|
| **Precision@k** | Of the k returned, what fraction were correct? | not wasting slots |
| **Recall@k** | Of all correct chunks, what fraction did we find? | completeness |
| **Hit@k** | Did we get *at least one* correct chunk? (0/1) | "did it work at all" |
| **MRR** | How high was the *first* correct chunk? (1/rank) | ranking quality |

In [ ]:
def precision_at_k(retrieved, gt, k):
    return 0.0 if k == 0 else len(set(retrieved[:k]) & gt) / k

def recall_at_k(retrieved, gt, k):
    return 0.0 if not gt else len(set(retrieved[:k]) & gt) / len(gt)

def hit_rate_at_k(retrieved, gt, k):
    return 1.0 if set(retrieved[:k]) & gt else 0.0

def reciprocal_rank(retrieved, gt):
    for i, idx in enumerate(retrieved):
        if idx in gt:
            return 1.0 / (i + 1)
    return 0.0

def evaluate_retriever(retrieve_fn, query_specs=QUERY_SPECS, k=K):
    """Run a retriever over all queries; return (per-query df, mean summary)."""
    rows = []
    for spec in query_specs:
        retrieved = [idx for idx, _ in retrieve_fn(spec["query"], k=k)]
        gt = spec["gt_indices"]
        rows.append({"query": spec["query"][:42],
                     "P@k": precision_at_k(retrieved, gt, k),
                     "R@k": recall_at_k(retrieved, gt, k),
                     "Hit": hit_rate_at_k(retrieved, gt, k),
                     "RR":  reciprocal_rank(retrieved, gt)})
    df_eval = pd.DataFrame(rows)
    summary = {"P@k": df_eval["P@k"].mean(), "R@k": df_eval["R@k"].mean(),
               "Hit": df_eval["Hit"].mean(), "MRR": df_eval["RR"].mean()}
    return df_eval, summary

print("Metrics + evaluate_retriever() ready. This harness scores EVERY retriever below.")

## 2.3 Bag of Words & TF-IDF — matching *words*

### 🧠 Concept
**Sparse / lexical** retrieval turns each chunk into a vector of word weights, then finds chunks
whose vector points the same way as the query's (**cosine similarity** = angle between vectors).

- **Bag of Words** = raw counts. Problem: `mg` appears in every dosage, so it looks "important".
- **TF-IDF** fixes this: weight = **T**erm **F**requency × **I**nverse **D**ocument **F**requency.
  A word scores high only if it's frequent *here* but rare *across the corpus* → `warfarin` beats `the`.

**🔑 The golden rule:** fit the vocabulary on **documents** (`fit_transform`), and only **apply** it
to the query (`transform`). Fitting on the query would corrupt the shared vocabulary — a classic bug.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --- Bag of Words, just to SEE the vocabulary ---
bow = CountVectorizer().fit(df_chunks["clean"])
bow_matrix = bow.transform(df_chunks["clean"])
freqs = np.asarray(bow_matrix.sum(axis=0)).flatten()
vocab = bow.get_feature_names_out()
top = freqs.argsort()[::-1][:10]
print(f"BoW matrix: {bow_matrix.shape} (chunks × vocab)")
print("Top terms by raw count:", ", ".join(f"{vocab[i]}({freqs[i]})" for i in top))

In [ ]:
# --- TF-IDF retriever (unigrams + bigrams) ---
tfidf_vec = TfidfVectorizer(ngram_range=(1, 2))
tfidf_matrix = tfidf_vec.fit_transform(df_chunks["clean"])   # DOCUMENTS → fit_transform
print(f"TF-IDF matrix: {tfidf_matrix.shape}")

def retrieve_top_k_tfidf(query, k=5):
    q_vec  = tfidf_vec.transform([preprocess_text(query, **PROFILES["readable_lemmatized"])])  # QUERY → transform
    scores = cosine_similarity(q_vec, tfidf_matrix).flatten()
    idx    = scores.argsort()[::-1][:k]
    return [(int(i), float(scores[i])) for i in idx if scores[i] > 0]

for idx, s in retrieve_top_k_tfidf("side effects of paracetamol", k=3):
    print(f"  [{idx:>2}] {s:.3f}  {df_chunks.loc[idx,'text'][:80]}")

In [ ]:
# --- Score TF-IDF on all 10 queries ---
tfidf_eval, tfidf_summary = evaluate_retriever(retrieve_top_k_tfidf)
print(tfidf_eval.to_string(index=False))
print(f"\nMEAN  P@{K}={tfidf_summary['P@k']:.3f}  R@{K}={tfidf_summary['R@k']:.3f}  "
      f"Hit@{K}={tfidf_summary['Hit']:.3f}  MRR={tfidf_summary['MRR']:.3f}")

## 2.4 Does the preprocessing profile matter? (Yes.)

### 🧠 Concept
We rebuild TF-IDF under each of the 4 profiles and score them. This *quantifies* the safety lesson
from Part 1: gentler cleaning that keeps negation/numbers is also **better retrieval** here.

In [ ]:
profile_scores = {}
for name, kw in PROFILES.items():
    texts = df_chunks["text"].apply(lambda t: preprocess_text(t, **kw))
    vec = TfidfVectorizer(ngram_range=(1, 2)).fit(texts)
    mat = vec.transform(texts)
    def make_fn(vec, mat, kw):
        def fn(query, k=5):
            q = vec.transform([preprocess_text(query, **kw)])
            sc = cosine_similarity(q, mat).flatten()
            idx = sc.argsort()[::-1][:k]
            return [(int(i), float(sc[i])) for i in idx if sc[i] > 0]
        return fn
    _, summ = evaluate_retriever(make_fn(vec, mat, kw))
    profile_scores[name] = summ

pd.DataFrame(profile_scores).T.round(3)

## 2.5 BM25 — the search-engine-grade upgrade

### 🧠 Concept
**BM25** is what real engines (Elasticsearch, Lucene) use. It improves TF-IDF with:
- **Saturation** — the 10th `mg` adds almost nothing (diminishing returns).
- **Length normalization** — long chunks don't win just for having more words.

We try the `rank_bm25` library; if it's missing we drop in a **tiny pure-NumPy BM25** so this
notebook *always* runs (and you get to see the formula).

In [ ]:
# --- BM25 with a self-contained fallback (educational + robust) ---
try:
    from rank_bm25 import BM25Okapi
    BM25_SOURCE = "rank_bm25 library"
except Exception:
    BM25_SOURCE = "built-in NumPy fallback"
    class BM25Okapi:
        """Minimal BM25 (Okapi) — same idea as the library, ~15 lines."""
        def __init__(self, corpus, k1=1.5, b=0.75):
            self.k1, self.b = k1, b
            self.corpus = corpus
            self.doc_len = np.array([len(d) for d in corpus])
            self.avgdl = self.doc_len.mean()
            self.N = len(corpus)
            df = Counter(t for doc in corpus for t in set(doc))       # docs containing term
            # BM25 idf with +1 smoothing (matches rank_bm25's behaviour closely)
            self.idf = {t: math.log(1 + (self.N - n + 0.5) / (n + 0.5)) for t, n in df.items()}
            self.tf = [Counter(doc) for doc in corpus]
        def get_scores(self, query):
            scores = np.zeros(self.N)
            for t in query:
                if t not in self.idf:
                    continue
                idf = self.idf[t]
                for i in range(self.N):
                    f = self.tf[i].get(t, 0)
                    if f == 0:
                        continue
                    denom = f + self.k1 * (1 - self.b + self.b * self.doc_len[i] / self.avgdl)
                    scores[i] += idf * (f * (self.k1 + 1)) / denom
            return scores

bm25_corpus = [text.split() for text in df_chunks["clean"]]
bm25 = BM25Okapi(bm25_corpus)
print(f"BM25 ready via: {BM25_SOURCE}")

def retrieve_top_k_bm25(query, k=5):
    q_tokens = preprocess_text(query, **PROFILES["readable_lemmatized"]).split()
    scores   = bm25.get_scores(q_tokens)
    idx      = np.argsort(scores)[::-1][:k]
    return [(int(i), float(scores[i])) for i in idx if scores[i] > 0]

bm25_eval, bm25_summary = evaluate_retriever(retrieve_top_k_bm25)
print(f"\nBM25  P@{K}={bm25_summary['P@k']:.3f}  R@{K}={bm25_summary['R@k']:.3f}  "
      f"Hit@{K}={bm25_summary['Hit']:.3f}  MRR={bm25_summary['MRR']:.3f}")

## 2.6 Where lexical retrieval *breaks*

### 🧠 Concept
Sparse retrieval can only match **words that literally appear**. Ask *"drugs safe during pregnancy"*
and — because no chunk contains that exact phrase — it returns loosely-related junk. This concrete
failure is the whole reason we need the next idea: **semantic** retrieval.

In [ ]:
hard = "drugs that are safe during pregnancy"
print(f'Query: "{hard}"\n')
for fn, name in [(retrieve_top_k_tfidf, "TF-IDF"), (retrieve_top_k_bm25, "BM25")]:
    print(f"--- {name} top 3 ---")
    for idx, s in fn(hard, k=3):
        print(f"  [{idx:>2}] {s:.3f}  {df_chunks.loc[idx,'drug']:<12s} {df_chunks.loc[idx,'text'][:70]}")
    print()
print("⚠️  No chunk literally says 'safe during pregnancy' → lexical retrieval cannot bridge the gap.")

---
# Part 3 — Lab 7: Dense (Semantic) & Hybrid Retrieval

## 3.1 Sentence embeddings — matching *meaning*

### 🧠 Concept
An **embedding model** maps text to a vector where *similar meanings land close together*, even with
**zero shared words**. So *"high blood sugar"* sits near *"diabetes" / "metformin"*. We encode all 84
chunks once, then compare each query's embedding to them by cosine similarity.

`normalize_embeddings=True` scales every vector to length 1, so cosine similarity becomes a plain
(fast) dot product.

> If `sentence-transformers` isn't installed or the model can't download, the next cell prints a clear
> message and disables the semantic/hybrid cells — the rest of the notebook still works.

In [ ]:
EMBEDDINGS_OK = True
try:
    from sentence_transformers import SentenceTransformer
    embed_model = SentenceTransformer(EMBED_MODEL_NAME)
    doc_embeddings = embed_model.encode(df_chunks["text"].tolist(),
                                        normalize_embeddings=True, show_progress_bar=False)
    print(f"Encoded {len(df_chunks)} chunks → shape {doc_embeddings.shape}")
except Exception as e:
    EMBEDDINGS_OK = False
    print(f"⚠️  Embeddings unavailable ({type(e).__name__}). Semantic/hybrid/FAISS cells will be skipped.")

In [ ]:
if EMBEDDINGS_OK:
    def retrieve_top_k_semantic(query, k=5):
        q = embed_model.encode([query], normalize_embeddings=True)
        scores = cosine_similarity(q, doc_embeddings).flatten()
        idx = scores.argsort()[::-1][:k]
        return [(int(i), float(scores[i])) for i in idx if scores[i] > 0]

    # Paraphrase test: NO keyword overlap with the corpus
    para = "medicine for high blood sugar"
    print(f'Query: "{para}"  (paraphrase — no exact match anywhere)\n')
    for idx, s in retrieve_top_k_semantic(para, k=3):
        print(f"  [{idx:>2}] {s:.3f}  {df_chunks.loc[idx,'drug']:<12s} {df_chunks.loc[idx,'text'][:80]}")
    print("\n✓ Semantic retrieval maps 'high blood sugar' → diabetes/metformin. Lexical could not.")
else:
    print("Skipped (embeddings unavailable).")

## 3.2 Every retriever, side by side

### 🧠 Concept
Now score the semantic retriever on the same 10 queries and stack it against TF-IDF and BM25.
Watch the trade-off emerge — no single method wins everything.

In [ ]:
rows = [{"Retriever": "TF-IDF", **tfidf_summary},
        {"Retriever": "BM25",   **bm25_summary}]
if EMBEDDINGS_OK:
    sem_eval, sem_summary = evaluate_retriever(retrieve_top_k_semantic)
    rows.append({"Retriever": "Embeddings", **sem_summary})
pd.DataFrame(rows).set_index("Retriever").round(3)

## 3.3 Dense retrieval has its *own* blind spots

### 🧠 Concept
Embeddings are fuzzy by design, so they're weak exactly where lexical is strong:
**exact numbers**, **negation**, and **abbreviations / exact drug codes**. Sparse and dense fail in
**opposite** situations — which is precisely why combining them works.

In [ ]:
if EMBEDDINGS_OK:
    for label, q in [("Numerics", "rated 10 out of 10 for effectiveness"),
                     ("Negation", "medications with no known side effects"),
                     ("Abbreviation", "drugs in the SSRI class")]:
        print(f'--- {label}: "{q}" ---')
        for idx, s in retrieve_top_k_semantic(q, k=3):
            print(f"  [{idx:>2}] {s:.3f}  {df_chunks.loc[idx,'drug']:<12s} field={df_chunks.loc[idx,'field']}")
        print()
else:
    print("Skipped (embeddings unavailable).")

## 3.4 Hybrid retrieval — the best of both

### 🧠 Concept
Combine lexical + semantic scores. Their raw scales differ, so we **min-max normalize** each to
`[0, 1]`, then take a weighted sum:

$$\text{hybrid} = \alpha \cdot \hat{s}_{\text{BM25}} + (1-\alpha)\cdot \hat{s}_{\text{emb}}$$

We use **α = 0.6** (lexical-heavy) because this data is full of exact tokens — drug names and
dosages — where BM25 shines, with embeddings adding paraphrase coverage.

In [ ]:
ALPHA = 0.6

def min_max_normalize(arr):
    mn, mx = arr.min(), arr.max()
    return np.ones_like(arr) * 0.5 if mx - mn < 1e-12 else (arr - mn) / (mx - mn)

if EMBEDDINGS_OK:
    def retrieve_top_k_hybrid(query, k=5):
        q_tokens  = preprocess_text(query, **PROFILES["readable_lemmatized"]).split()
        bm25_norm = min_max_normalize(bm25.get_scores(q_tokens))
        q_emb     = embed_model.encode([query], normalize_embeddings=True)
        emb_norm  = min_max_normalize(cosine_similarity(q_emb, doc_embeddings).flatten())
        hybrid    = ALPHA * bm25_norm + (1 - ALPHA) * emb_norm
        idx       = hybrid.argsort()[::-1][:k]
        return [(int(i), float(hybrid[i])) for i in idx]

    hybrid_eval, hybrid_summary = evaluate_retriever(retrieve_top_k_hybrid)
    print(hybrid_eval.to_string(index=False))
    print(f"\nHYBRID  P@{K}={hybrid_summary['P@k']:.3f}  R@{K}={hybrid_summary['R@k']:.3f}  "
          f"Hit@{K}={hybrid_summary['Hit']:.3f}  MRR={hybrid_summary['MRR']:.3f}")
else:
    # Fall back to BM25 as the "hybrid" so downstream RAG still runs
    retrieve_top_k_hybrid = retrieve_top_k_bm25
    hybrid_summary = bm25_summary
    print("Embeddings unavailable → using BM25 as the retrieval backend for the RAG demo.")

## 3.5 FAISS — making it scale

### 🧠 Concept
With 84 chunks, brute-force cosine is instant. But at **millions** of vectors you need an index.
**FAISS** is that library. `IndexFlatIP` does inner-product search = cosine on our normalized
vectors. Same math, built to scale. (Optional — skipped if FAISS isn't installed.)

In [ ]:
if EMBEDDINGS_OK:
    try:
        import faiss
        dim = doc_embeddings.shape[1]
        faiss_index = faiss.IndexFlatIP(dim)
        faiss_index.add(doc_embeddings.astype("float32"))
        q = embed_model.encode(["How does metformin work?"], normalize_embeddings=True)
        D, I = faiss_index.search(q.astype("float32"), 3)
        print(f"FAISS IndexFlatIP: {faiss_index.ntotal} vectors, dim={dim}\n")
        for rank, (idx, d) in enumerate(zip(I[0], D[0]), 1):
            print(f"  rank {rank}: [{idx}] cos={d:.3f}  {df_chunks.loc[idx,'text'][:75]}")
    except Exception as e:
        print(f"FAISS not available ({type(e).__name__}) — brute-force cosine already works for 84 chunks.")
else:
    print("Skipped (embeddings unavailable).")

---
# Part 4 — Lab 8: RAG Assembly (Context + Prompt)

## 4.1 Pack the context — give the LLM *citable* sources

### 🧠 Concept
We hand the retrieved chunks to the LLM as **numbered, labelled source blocks**. The numbering
(`[Source 1]`, `[Source 2]`, …) is what lets the model cite, and lets *us* verify the answer against
the exact chunk it used.

In [ ]:
def get_chunk_metadata(idx):
    r = df_chunks.loc[idx]
    return {"drug": r["drug"], "category": r["category"], "field": r["field"], "idx": int(idx)}

def pack_context(retrieved_results):
    parts = []
    for rank, (idx, score) in enumerate(retrieved_results, start=1):
        m = get_chunk_metadata(idx)
        parts.append(f'[Source {rank}] (drug={m["drug"]}, field={m["field"]}, score={score:.3f})\n'
                     f'{df_chunks.loc[idx, "text"]}')
    return "\n\n".join(parts)

print("pack_context() ready.")

## 4.2 The prompt — where "no hallucination" is enforced

### 🧠 Concept
The system prompt is the **contract**. Everything upstream (safe cleaning, good retrieval) exists to
fill this prompt with accurate context; the *rules* then bind the model to it: use only the context,
cite sources, keep numbers and negations, add the disclaimer, and **refuse** when the context is
insufficient.

In [ ]:
SYSTEM_PROMPT = """You are a pharmaceutical information assistant. Your role is to answer
drug-related questions using ONLY the provided context sources.

RULES:
1. Base your answer STRICTLY on the provided context. Do not use outside knowledge.
2. Cite sources inline using [Source N] notation.
3. If the context does not contain enough information, say so explicitly.
4. NEVER provide medical advice. Always include: "This is not medical advice.
   Consult a healthcare professional for medical decisions."
5. Preserve exact numbers (dosages, frequencies) and negation words (no, not, never).
6. If the question cannot be answered from the context, refuse with an explanation."""

def build_grounded_prompt(query, context):
    return f"""{SYSTEM_PROMPT}

---
CONTEXT:
{context}
---

QUESTION: {query}

ANSWER (include [Source N] citations and the medical disclaimer):"""

# See a full assembled prompt
demo_q = "What are the side effects of paracetamol?"
demo_context = pack_context(retrieve_top_k_hybrid(demo_q, k=K))
print(build_grounded_prompt(demo_q, demo_context)[:1200], "\n... (truncated)")

---
# Part 5 — Lab 9: Grounded Generation with a Local LLM

## 5.1 Talk to Ollama (local, private, free)

### 🧠 Concept
We run the LLM **locally** via [Ollama](https://ollama.com) so patient-style data never leaves the
machine, and it's free. `temperature=0.0` makes generation deterministic — we want facts, not
creativity. The health check means the notebook still completes if Ollama isn't running.

> To enable real answers: install Ollama, then `ollama pull deepseek-r1:1.5b` and keep it running.

In [ ]:
import requests

def ollama_status():
    try:
        r = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=5); r.raise_for_status()
        models = [m["name"] for m in r.json().get("models", [])]
        has = any(OLLAMA_MODEL in m for m in models)
        print(f"Ollama OK at {OLLAMA_HOST} | models={models} | {OLLAMA_MODEL} present={has}")
        return has
    except Exception as e:
        print(f"Ollama NOT reachable ({e}). Generation cells will be skipped.")
        return False

OLLAMA_OK = ollama_status()

In [ ]:
def ask_ollama(prompt, temperature=0.0, timeout=180):
    r = requests.post(f"{OLLAMA_HOST}/api/generate",
                      json={"model": OLLAMA_MODEL, "prompt": prompt,
                            "stream": False, "options": {"temperature": temperature}},
                      timeout=timeout)
    r.raise_for_status()
    return r.json()["response"]

def extract_final_answer(raw):
    """DeepSeek-R1 emits <think>...</think> reasoning — strip it, keep the answer."""
    cleaned = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip()
    return cleaned or raw.strip()

def rag_answer(query, k=5):
    """The full pipeline in five lines: retrieve → pack → prompt → generate → clean."""
    results = retrieve_top_k_hybrid(query, k=k)
    context = pack_context(results)
    prompt  = build_grounded_prompt(query, context)
    raw     = ask_ollama(prompt)
    return extract_final_answer(raw), results

print("ask_ollama(), extract_final_answer(), rag_answer() ready.")

## 5.2 Run the full RAG loop on real questions

In [ ]:
DEMO_QUESTIONS = [
    "What are the side effects of paracetamol?",
    "How does metformin work to treat diabetes?",
    "Can I take ibuprofen if I am on warfarin?",
    "What is the recommended dosage for amoxicillin in adults?",
]

if OLLAMA_OK:
    for i, q in enumerate(DEMO_QUESTIONS, 1):
        print("=" * 78)
        print(f"Q{i}: {q}")
        print("=" * 78)
        answer, results = rag_answer(q, k=K)
        print("Sources used:")
        for idx, s in results:
            print(f"  [{idx:>2}] {s:.3f}  {df_chunks.loc[idx,'drug']:<12s} field={df_chunks.loc[idx,'field']}")
        print("\nAnswer:\n" + answer + "\n")
else:
    print("Ollama offline → skipping generation. Retrieval + prompt assembly above already work.")
    # Still show what WOULD be sent, to prove the pipeline:
    q = DEMO_QUESTIONS[0]
    print(f'\nExample assembled context for "{q}":\n')
    print(pack_context(retrieve_top_k_hybrid(q, k=3)))

---
# Part 6 — Evaluation & Failure Analysis

## 6.1 Retrieval scoreboard

### 🧠 Concept
Line up every retriever's mean metrics. On this structured, exact-token-heavy data, **hybrid**
typically leads — it inherits BM25's precision on drug names *and* embeddings' paraphrase reach.

In [ ]:
rows = [{"Retriever": "TF-IDF", **tfidf_summary}, {"Retriever": "BM25", **bm25_summary}]
if EMBEDDINGS_OK:
    rows.append({"Retriever": "Embeddings", **sem_summary})
    rows.append({"Retriever": "Hybrid",     **hybrid_summary})
board = pd.DataFrame(rows).set_index("Retriever").round(3)
print(board.to_string())
print(f"\nBest MRR   : {board['MRR'].idxmax()} ({board['MRR'].max():.3f})")
print(f"Best Hit@{K}: {board['Hit'].idxmax()} ({board['Hit'].max():.3f})")

## 6.2 Groundedness check — did the answer behave?

### 🧠 Concept
A RAG answer is only trustworthy if it (a) **cites its sources** and (b) carries the **medical
disclaimer**. We verify both mechanically on the last generated answer.

In [ ]:
if OLLAMA_OK:
    answer, _ = rag_answer(DEMO_QUESTIONS[0], k=K)
    citations  = len(re.findall(r"\[Source \d+\]", answer))
    disclaimer = "not medical advice" in answer.lower()
    print(f"Source citations found : {citations}")
    print(f"Medical disclaimer     : {'YES' if disclaimer else 'NO'}")
    print(f"Answer length (words)  : {len(answer.split())}")
else:
    print("No generation available — start Ollama to run the groundedness check.")

## 6.3 The refusal test — the most important safety check

### 🧠 Concept
A good RAG system is defined as much by what it **refuses** as by what it answers. Our data has **no
pricing field**, so *"What is the price of aspirin in Egypt?"* is unanswerable. The correct behaviour
is to **say so** — never invent a price. This is where hallucination would be most dangerous, so it's
the test that matters most.

In [ ]:
unanswerable = "What is the price of aspirin in Egypt?"
print(f'Unanswerable query: "{unanswerable}"\n')
print("Retrieval still returns *something* (its nearest chunks) — that is expected:")
for idx, s in retrieve_top_k_hybrid(unanswerable, k=3):
    print(f"  [{idx:>2}] {s:.3f}  {df_chunks.loc[idx,'text'][:80]}")

if OLLAMA_OK:
    ans, _ = rag_answer(unanswerable)
    print("\nAnswer:\n" + ans)
    refused = any(w in ans.lower()[:80] for w in ["no", "not", "cannot", "does not", "doesn't"])
    print("\n✓ Correctly signalled it cannot answer." if refused
          else "\n⚠️  Check for hallucination — it must NOT invent a price.")
else:
    print("\nExpected behaviour: refuse / state the context has no pricing information.")

## 6.4 Pipeline complete — the cheat sheet

You just built a full RAG system from scratch. Keep this mental model:

```
CHUNK      → make each fact searchable in isolation (+ metadata for citation)
CLEAN      → normalize words, but NEVER lose numbers or negation (safety > tidiness)
RETRIEVE   → sparse = words · dense = meaning · hybrid = both; always MEASURE (P / R / Hit / MRR)
PACK+PROMPT→ hand the LLM ONLY trusted, numbered sources + strict rules
GENERATE   → LLM rephrases + cites facts, and REFUSES when the context can't answer
EVALUATE   → prove groundedness (citations + disclaimer) AND prove refusal
```

**The five lessons that matter most**
1. Retrieval quality caps everything — the LLM can't cite what search didn't find.
2. Aggressive preprocessing is *dangerous* for medical text (it deletes dosages and negations).
3. Sparse and dense retrieval fail in **opposite** cases → hybrid beats either alone.
4. `documents → fit_transform`, `query → transform` — never fit on the query.
5. A trustworthy system is judged by its **refusals**, not just its answers.

In [ ]:
print("=" * 66)
print("  PhARMA RAG — LEARNING EDITION — COMPLETE")
print("=" * 66)
print(f"Dataset    : {len(drugs_data)} drugs × {len(FIELD_LABELS)} fields = {len(df_chunks)} chunks")
print(f"Retrievers : TF-IDF, BM25 ({BM25_SOURCE})"
      + (", Embeddings, Hybrid(a=0.6)" if EMBEDDINGS_OK else "  [embeddings offline]"))
print(f"LLM        : {OLLAMA_MODEL} via Ollama ({'connected' if OLLAMA_OK else 'offline'})")
print(f"Evaluation : {len(QUERY_SPECS)} queries × 4 metrics (P@{K}, R@{K}, Hit@{K}, MRR)")
print("\nDone. Re-run any section and tweak ALPHA, K, or the PROFILES to see how scores move.")